<a href="https://colab.research.google.com/github/manasamorthad/DeepLearning/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#next character


In [3]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Input
import numpy as np

# Define vocabulary
vocab = ['c', 'a', 't']
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

# Training sequence
sequence = ['c', 'a']
target = ['a', 't']   # next characters

# One-hot encoding
X = np.eye(len(vocab))[[char2idx[c] for c in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[char2idx[c] for c in target]]

# Build model
model = Sequential([
    Input(shape=(1, len(vocab))),
    SimpleRNN(8),
    Dense(len(vocab), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train model
model.fit(X, y, epochs=200, verbose=0)

# Predict next character after 'a'
test = np.eye(len(vocab))[char2idx['a']].reshape(1, 1, len(vocab))
pred = model.predict(test, verbose=0)

print("Next char prediction:", idx2char[np.argmax(pred)])

Next char prediction: t


#next word

In [5]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Input
import numpy as np

vocab = ['sun', 'is', 'hot', '<stop>']
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

sequence = ['sun', 'is', 'hot']
target = ['is', 'hot', '<stop>']

X = np.eye(len(vocab))[[word2idx[w] for w in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[word2idx[w] for w in target]]

model = Sequential([
    Input(shape=(1, len(vocab))),
    SimpleRNN(8),
    Dense(len(vocab), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.fit(X, y, epochs=300, verbose=0)

test = np.eye(len(vocab))[word2idx['is']].reshape(1, 1, len(vocab))
pred = model.predict(test, verbose=0)

print("Next word prediction:", idx2word[np.argmax(pred)])

Next word prediction: hot


#sentence

In [6]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Input
import numpy as np

vocab = ['i am happy', 'i go school', 'i learn well', '<stop>']
sent2idx = {s: i for i, s in enumerate(vocab)}
idx2sent = {i: s for s, i in sent2idx.items()}

sequence = ['i am happy', 'i go school', 'i learn well']
target = ['i go school', 'i learn well', '<stop>']

X = np.eye(len(vocab))[[sent2idx[s] for s in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[sent2idx[s] for s in target]]

model = Sequential([
    Input(shape=(1, len(vocab))),
    SimpleRNN(8),
    Dense(len(vocab), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.fit(X, y, epochs=400, verbose=0)

test = np.eye(len(vocab))[sent2idx['i go school']].reshape(1, 1, len(vocab))
pred = model.predict(test, verbose=0)

print("Next sentence prediction:", idx2sent[np.argmax(pred)])

Next sentence prediction: i learn well


#LSTM


In [8]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np

vocab = {'we': 0, 'like': 1, 'coding': 2, 'daily': 3, 'fun': 4, 'learn': 5}
idx2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

X = np.array([[0, 1, 2, 3]])
y = np.array([4])

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=10, input_length=4),
    LSTM(32),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(X, y, epochs=200, verbose=0)

test = np.array([[0, 1, 2, 3]])
pred = model.predict(test, verbose=0)

print("LSTM Predicted next word:", idx2word[np.argmax(pred)])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


LSTM Predicted next word: fun


#GRU

In [9]:
from keras.models import Sequential
from keras.layers import Embedding, GRU, Dense
import numpy as np

vocab = {'he': 0, 'plays': 1, 'football': 2, 'well': 3, 'goal': 4, 'team': 5}
idx2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

X = np.array([[0, 1, 2, 3]])
y = np.array([4])

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=10, input_length=4),
    GRU(32),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(X, y, epochs=200, verbose=0)

test = np.array([[0, 1, 2, 3]])
pred = model.predict(test, verbose=0)

print("GRU Predicted next word:", idx2word[np.argmax(pred)])

GRU Predicted next word: goal


#Encoder-Decoder

In [10]:
from keras.models import Model
from keras.layers import Input, SimpleRNN, Dense
import numpy as np

input_text = "good morning"
target_text = "\tశుభోదయం\n"

input_chars = sorted(set(input_text))
target_chars = sorted(set(target_text))

input_idx = {ch: i for i, ch in enumerate(input_chars)}
target_idx = {ch: i for i, ch in enumerate(target_chars)}
reverse_target_idx = {i: ch for ch, i in target_idx.items()}

encoder_input_data = np.zeros((1, len(input_text), len(input_chars)))
decoder_input_data = np.zeros((1, len(target_text)-1, len(target_chars)))
decoder_target_data = np.zeros((1, len(target_text)-1, len(target_chars)))

for t, ch in enumerate(input_text):
    encoder_input_data[0, t, input_idx[ch]] = 1

for t, ch in enumerate(target_text[:-1]):
    decoder_input_data[0, t, target_idx[ch]] = 1

for t, ch in enumerate(target_text[1:]):
    decoder_target_data[0, t, target_idx[ch]] = 1

enc_in = Input(shape=(None, len(input_chars)))
_, enc_state = SimpleRNN(64, return_state=True)(enc_in)

dec_in = Input(shape=(None, len(target_chars)))
dec_out, _ = SimpleRNN(64, return_sequences=True, return_state=True)(dec_in, initial_state=enc_state)
dec_out = Dense(len(target_chars), activation='softmax')(dec_out)

model = Model([enc_in, dec_in], dec_out)
model.compile(optimizer='adam', loss='categorical_crossentropy')
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, epochs=300, verbose=0)

out = model.predict([encoder_input_data, decoder_input_data], verbose=0)
translated = ''.join([reverse_target_idx[np.argmax(vec)] for vec in out[0]])
translated = translated.replace('\t','').replace('\n','')

print("Input:", input_text)
print("Predicted:", translated)

Input: good morning
Predicted: శుభోదయం


#Attention Mechanism

In [14]:
from keras.models import Model
from keras.layers import Input, Embedding, SimpleRNN, Dense, Attention, GlobalAveragePooling1D
import numpy as np

vocab = {'movie': 0, 'is': 1, 'good': 2, 'bad': 3}
idx2label = {0: 'negative', 1: 'positive'}
vocab_size = len(vocab)

X = np.array([
    [0, 1, 2],
    [0, 1, 3]
])

y = np.array([1, 0])

inputs = Input(shape=(3,))
embed = Embedding(input_dim=vocab_size, output_dim=8)(inputs)
rnn_out = SimpleRNN(16, return_sequences=True)(embed)

attention_out = Attention()([rnn_out, rnn_out])
pooled = GlobalAveragePooling1D()(attention_out)

outputs = Dense(2, activation='softmax')(pooled)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=0)

test = np.array([[0, 1, 2]])
pred = model.predict(test, verbose=0)

print("Predicted class:", idx2label[np.argmax(pred)])


Predicted class: positive
